In [2]:
import os
import pickle
import numpy as np

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
MODEL_PATH = "models/mailmind_email_bilstm_v1.keras"

SENDER_TOKENIZER_PATH = "tokenizers/sender_tokenizer.pkl"
SUBJECT_TOKENIZER_PATH = "tokenizers/subject_tokenizer.pkl"
BODY_TOKENIZER_PATH = "tokenizers/body_tokenizer.pkl"
LABEL_ENCODER_PATH = "tokenizers/email_label_encoder.pkl"

print("Artifact paths configured.")

In [ ]:
model = load_model(MODEL_PATH)

print("Model loaded successfully.")
print("Model name:", model.name)

In [ ]:
with open(SENDER_TOKENIZER_PATH, "rb") as f:
    sender_tokenizer = pickle.load(f)

with open(SUBJECT_TOKENIZER_PATH, "rb") as f:
    subject_tokenizer = pickle.load(f)

with open(BODY_TOKENIZER_PATH, "rb") as f:
    body_tokenizer = pickle.load(f)

with open(LABEL_ENCODER_PATH, "rb") as f:
    label_encoder = pickle.load(f)

print("Tokenizers loaded.")
print("Label encoder loaded.")

print("\nClasses:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{i} -> {label}")

In [ ]:
SENDER_MAX_LEN = 50
SUBJECT_MAX_LEN = 100
BODY_MAX_LEN = 400

print("Sender max length :", SENDER_MAX_LEN)
print("Subject max length:", SUBJECT_MAX_LEN)
print("Body max length   :", BODY_MAX_LEN)

In [ ]:
def predict_email(sender, subject, body):

    # Sender
    sender_seq = sender_tokenizer.texts_to_sequences([sender])

    sender_pad = pad_sequences(
        sender_seq,
        maxlen=SENDER_MAX_LEN,
        padding="post",
        truncating="post"
    )

    # Subject
    subject_seq = subject_tokenizer.texts_to_sequences([subject])

    subject_pad = pad_sequences(
        subject_seq,
        maxlen=SUBJECT_MAX_LEN,
        padding="post",
        truncating="post"
    )

    # Body
    body_seq = body_tokenizer.texts_to_sequences([body])

    body_pad = pad_sequences(
        body_seq,
        maxlen=BODY_MAX_LEN,
        padding="post",
        truncating="post"
    )

    # Prediction
    probabilities = model.predict(
        [
            sender_pad,
            subject_pad,
            body_pad
        ],
        verbose=0
    )[0]

    predicted_index = np.argmax(probabilities)

    predicted_label = label_encoder.inverse_transform(
        [predicted_index]
    )[0]

    confidence = probabilities[predicted_index]

    return {
        "label": predicted_label,
        "confidence": float(confidence),
        "probabilities": {
            label_encoder.classes_[i]: float(probabilities[i])
            for i in range(len(label_encoder.classes_))
        }
    }

In [ ]:
def display_prediction(result):

    print("MailMind Classification Result")
    print("=" * 50)

    print(f"Predicted Label : {result['label']}")
    print(f"Confidence      : {result['confidence']:.2%}")

    print("\nClass Probabilities:")

    for label, probability in result["probabilities"].items():
        print(f"  {label:<10}: {probability:.2%}")

    print("=" * 50)

In [ ]:
result_1 = predict_email(

    sender="cdc@vit.ac.in",

    subject="Microsoft Placement Drive – Registration Open",

    body="""
Dear Students,

Microsoft will be visiting the VIT campus for the upcoming placement drive.

Eligible students are requested to register before the deadline.

The online assessment schedule and eligibility criteria will be shared
with the shortlisted students.

Regards,
Career Development Centre
"""
)

display_prediction(result_1)

In [ ]:
result_2 = predict_email(

    sender="faculty.cse@vit.ac.in",

    subject="Operating Systems – Module 3 Materials",

    body="""
Dear Students,

The study materials for Module 3 of Operating Systems have been uploaded
to the course portal.

Students are requested to refer to the materials before the upcoming
classes.

Regards,
Department of Computer Science
"""
)

display_prediction(result_2)

In [ ]:
result_3 = predict_email(

    sender="studentservices@vit.ac.in",

    subject="Action Required: Update Student Information",

    body="""
Dear Student,

All students are required to verify and update their contact information
on the student portal before August 31.

Please complete the process before the deadline to avoid issues with
university communication.

Regards,
Student Services
"""
)

display_prediction(result_3)

In [ ]:
result_4 = predict_email(

    sender="offers@wealthgrowth-now.com",

    subject="Congratulations! You Have Been Selected",

    body="""
Congratulations!

You have been selected to receive an exclusive cash reward of $25,000.

Click the link below immediately to claim your prize.
This limited offer expires today.

Regards,
Rewards Department
"""
)

display_prediction(result_4)

In [ ]:
result_5 = predict_email(

    sender="loanapproval@quickcash-finance.com",

    subject="Your Loan Has Been Pre-Approved",

    body="""
Dear Customer,

Your personal loan of $50,000 has been pre-approved with no credit check.

To receive your money today, submit your bank account information using
the link below.

Limited-time processing offer available now.

Regards,
QuickCash Finance
"""
)

display_prediction(result_5)

In [ ]:
result_6 = predict_email(

    sender="'Helpdesk CDC' via VIT Placement Group <cdc@vit.ac.in>",

    subject="Urgent: VISA Online Test is Scheduled on 8th October 2025",

    body="""
Kind Attention!!

The VISA online test is scheduled on 8th October 2025 by 4.30 PM
at PRP717, Pearl Research Park.

Please find the shortlisted student list attached.

All shortlisted students are requested to report on time and carry
their university ID card.

Regards,
Career Development Centre
"""
)

display_prediction(result_6)

In [ ]:
demo_results = [
    ("Microsoft Placement", result_1),
    ("Academic Materials", result_2),
    ("Student Information", result_3),
    ("Reward Scam", result_4),
    ("Fake Loan Offer", result_5),
    ("VISA Online Test", result_6)
]

for name, result in demo_results:

    print(
        f"{name:<25} -> "
        f"{result['label']:<10} "
        f"({result['confidence']:.2%})"
    )